<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
def load_data(seed=42):  # IMPORTANTE: seed com valor padrão
    X, y = fetch_openml("mnist_784", as_frame=False, return_X_y=True)
    
    y = y.astype(int)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    
    return X_train, X_test, y_train, y_test



# RESPOSTA:
# Não é necessário normalizar os dados para Random Forest e AdaBoost,
# pois esses modelos são baseados em árvores de decisão, que não dependem da escala dos dados.
# Eles fazem divisões com base em limiares (thresholds), e não utilizam distância.
# Ainda assim, normalizar pode ser útil em pipelines mais gerais

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def train_random_forest(X_train, y_train, seed=42):  # IMPORTANTE
    model = RandomForestClassifier(random_state=seed)
    model.fit(X_train, y_train)
    return model

def train_adaboost(X_train, y_train, seed=42):  # IMPORTANTE
    model = AdaBoostClassifier(random_state=seed)
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [ ]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)


# RESPOSTA:
# A função realiza a predição no conjunto de teste e calcula a acurácia,
# que representa a proporção de acertos do modelo.

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [ ]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)
    
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("Modelo inválido")
    
    return evaluate(model, X_test, y_test)

# RESPOSTA:
# O overfitting começa quando a árvore cresce demais e passa a memorizar os dados de treino,
# capturando ruídos ao invés de padrões gerais.
#
# Quando max_depth=None, a árvore cresce até separar completamente os dados,
# permitindo atingir 100% de acurácia no treino, pois cada folha pode representar
# exatamente subconjuntos específicos dos dados.

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def full_evaluation(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted"),
        "recall": recall_score(y_test, y_pred, average="weighted"),
        "f1": f1_score(y_test, y_pred, average="weighted")
    }

X_train, X_test, y_train, y_test = load_data(42)

rf = train_random_forest(X_train, y_train, 42)
ab = train_adaboost(X_train, y_train, 42)

rf_metrics = full_evaluation(rf, X_test, y_test)
ab_metrics = full_evaluation(ab, X_test, y_test)

print("Random Forest:", rf_metrics)
print("AdaBoost:", ab_metrics)



# RESPOSTA:
# O Random Forest geralmente apresenta melhor desempenho inicial,
# pois reduz a variância ao combinar várias árvores (bagging).
# Já o AdaBoost foca nos erros e pode ser mais sensível a ruídos.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [ ]:
for seed in [42, 7]:
    print("Seed:", seed)
    print("RF:", run_pipeline("rf", seed))
    print("AB:", run_pipeline("ab", seed))



# RESPOSTA:
# Os resultados podem variar entre diferentes seeds, pois a divisão dos dados muda.
# O experimento é reprodutível porque o uso de random_state garante
# que, para a mesma seed, os resultados serão sempre os mesmos.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
X_train, X_test, y_train, y_test = load_data(42)

rf = train_random_forest(X_train, y_train, 42)

train_acc = accuracy_score(y_train, rf.predict(X_train))
test_acc = accuracy_score(y_test, rf.predict(X_test))

print("Train:", train_acc)
print("Test:", test_acc)


# RESPOSTA:
# Existe overfitting quando a acurácia no treino é significativamente maior que no teste.
# O Random Forest reduz overfitting em relação a uma árvore única,
# mas ainda pode sofrer com isso dependendo dos parâmetros.
# O AdaBoost também pode sofrer overfitting, especialmente em dados com ruído.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
for n in [10, 50, 100]:
    model = RandomForestClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    print("RF n_estimators =", n, evaluate(model, X_test, y_test))

for n in [10, 50, 100]:
    model = AdaBoostClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    print("AB n_estimators =", n, evaluate(model, X_test, y_test))

# RESPOSTA:
# O desempenho pode melhorar com mais estimadores, mas tende a estabilizar após certo ponto.
# O AdaBoost costuma ser mais sensível a mudanças em n_estimators,
# enquanto o Random Forest é mais estável devido à média entre várias árvores.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

In [ ]:
# 1:
# A acurácia não é suficiente, pois pode mascarar problemas como desbalanceamento de classes.
# Métricas como precisão, recall e F1-score fornecem uma avaliação mais completa.

# 2:
# Para garantir que o resultado não ocorreu por acaso, utilizamos random_state
# e testamos diferentes seeds, verificando a estabilidade dos resultados.

# 3:
# Dois problemas metodológicos são:
# (1) uso de apenas uma divisão treino/teste
# (2) ausência de validação cruzada

# 4:
# O pipeline é relativamente confiável, pois garante reprodutibilidade e separação adequada dos dados.
# Porém, sua confiabilidade é limitada pela ausência de validação cruzada
# e pela dependência de um único conjunto de teste.